# Band Cut Grid Search (Colab)
Grid search over band cutoffs using only config/model definitions (no dataset_*.py dependency).

In [ ]:

import os
import sys
import shutil
from pathlib import Path


use_colab = "google.colab" in sys.modules
if use_colab:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    project_dir = Path('/content/drive/MyDrive/liveness_detection_vae')
else:
    project_dir = Path.cwd()


In [ ]:

import os
import sys
import shutil
from pathlib import Path

# project_dir is set in the previous cell
lib_dir = project_dir / 'lib'
sys.path.insert(0, str(lib_dir))
sys.path.insert(0, str(project_dir))

drive_root = project_dir / 'datasets'
drive_zip = drive_root / f"{DATASET_NAME}.zip"
local_root = Path('/content/datasets') if "google.colab" in sys.modules else drive_root
if "google.colab" in sys.modules:
    local_root.mkdir(parents=True, exist_ok=True)
local_zip = local_root / f"{DATASET_NAME}.zip"
extract_dir = local_root / DATASET_NAME
nested_dir = extract_dir / DATASET_NAME

def _has_npz(p: Path) -> bool:
    return p.is_dir() and any(p.rglob('*.npz'))

# Sync zip to local if available
if drive_zip.exists() and (not local_zip.exists() or drive_zip.stat().st_mtime > local_zip.stat().st_mtime):
    shutil.copy2(drive_zip, local_zip)

# Resolve data_dir (prefer local extracted; else extract local zip; else drive extracted)
if _has_npz(extract_dir):
    data_dir = extract_dir
elif _has_npz(nested_dir):
    data_dir = nested_dir
elif local_zip.exists():
    shutil.unpack_archive(str(local_zip), str(local_root))
    if _has_npz(nested_dir):
        data_dir = nested_dir
    elif _has_npz(extract_dir):
        data_dir = extract_dir
    else:
        data_dir = None
else:
    data_dir = None

if data_dir is None:
    drive_extract = drive_root / DATASET_NAME
    drive_nested = drive_extract / DATASET_NAME
    if _has_npz(drive_nested):
        data_dir = drive_nested
    elif _has_npz(drive_extract):
        data_dir = drive_extract

if data_dir is None:
    raise FileNotFoundError(f"Expected {DATASET_NAME} zip/extracted under {drive_root} or local {local_root}")

os.environ['BANDVAE_DATA_DIR'] = str(data_dir)

split_json = project_dir / 'data_split.json'
base_dir = data_dir if (data_dir / 'test_balanced_npz').exists() else data_dir.parent
os.chdir(project_dir)
print('Split JSON:', split_json)
print(f'Project dir: {project_dir}')
print(f'Data dir: {data_dir}')


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

sys.path.append(str(project_dir))
from config_bandvae import Config
from model_bandvae import BandSplitVAE, band_split_vae_loss

In [ ]:

import json
import numpy as np
import torch
from pathlib import Path
from scipy.signal import butter, filtfilt
from torch.utils.data import Dataset

class InlineStage2Dataset(Dataset):
    """Minimal Stage2 dataset without external dataset_*.py dependency."""
    def __init__(self, split_json_path, split_name, T_fixed=300, fps=30,
                 use_acceleration=True, use_angle=True, use_angle_rate=True,
                 fc_low=2.0, fc_high=8.0, filter_order=4, random_crop=True,
                 base_dir=None):
        self.T_fixed = T_fixed
        self.fps = fps
        self.use_acceleration = use_acceleration
        self.use_angle = use_angle
        self.use_angle_rate = use_angle_rate
        self.fc_low = fc_low
        self.fc_high = fc_high
        self.filter_order = filter_order
        self.random_crop = random_crop
        self.base_dir = Path(base_dir) if base_dir else None

        with open(split_json_path, 'r') as f:
            split_data = json.load(f)[split_name]
        self.files = [self._resolve_path(p) for p in split_data['real']] + [self._resolve_path(p) for p in split_data['fake']]
        self.labels = [0] * len(split_data['real']) + [1] * len(split_data['fake'])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        lips_outer = data['lips_outer']
        lips_inner = data['lips_inner']
        landmarks = np.concatenate([lips_outer[:, :-1, :], lips_inner[:, :-1, :]], axis=1)

        h, w = data['size']
        landmarks = landmarks.copy()
        landmarks[..., 0] /= w + 1e-8
        landmarks[..., 1] /= h + 1e-8

        features = self._build_features(landmarks)
        features = self._crop_or_pad(features)
        x_lf, x_bp, x_hf = self._apply_filters(features)

        x_lf = torch.from_numpy(x_lf).float()
        x_bp = torch.from_numpy(x_bp).float()
        x_hf = torch.from_numpy(x_hf).float()
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return x_lf, x_bp, x_hf, label

    def _build_features(self, landmarks):
        T, N, _ = landmarks.shape
        position = landmarks
        velocity = np.zeros_like(position)
        velocity[1:] = np.diff(position, axis=0) * self.fps

        feature_list = [position, velocity]

        if self.use_acceleration:
            acceleration = np.zeros_like(position)
            acceleration[1:] = np.diff(velocity, axis=0) * self.fps
            feature_list.append(acceleration)

        if self.use_angle or self.use_angle_rate:
            angle = np.arctan2(landmarks[:, :, 1], landmarks[:, :, 0])
            angle = np.expand_dims(angle, axis=-1)
            if self.use_angle:
                feature_list.append(angle)
            if self.use_angle_rate:
                angle_rate = np.zeros_like(angle)
                angle_rate[1:] = np.diff(angle, axis=0) * self.fps
                feature_list.append(angle_rate)

        return np.concatenate(feature_list, axis=-1)

    def _crop_or_pad(self, features):
        T, N, C = features.shape
        if T == self.T_fixed:
            return features
        if T > self.T_fixed:
            start = 0 if not self.random_crop else np.random.randint(0, T - self.T_fixed + 1)
            return features[start : start + self.T_fixed]
        pad = np.zeros((self.T_fixed - T, N, C), dtype=features.dtype)
        return np.concatenate([features, pad], axis=0)

    def _apply_filters(self, features):
        T, N, C = features.shape
        x_lf = np.zeros((T, N, C), dtype=np.float32)
        x_bp = np.zeros((T, N, C), dtype=np.float32)
        x_hf = np.zeros((T, N, C), dtype=np.float32)

        b_lf, a_lf = butter(self.filter_order, self.fc_low / (0.5 * self.fps), btype='low', analog=False)
        b_bp, a_bp = butter(self.filter_order, [self.fc_low / (0.5 * self.fps), self.fc_high / (0.5 * self.fps)], btype='band')
        b_hf, a_hf = butter(self.filter_order, self.fc_high / (0.5 * self.fps), btype='high', analog=False)

        for n in range(N):
            for c in range(C):
                signal = features[:, n, c]
                x_lf[:, n, c] = filtfilt(b_lf, a_lf, signal)
                x_bp[:, n, c] = filtfilt(b_bp, a_bp, signal)
                x_hf[:, n, c] = filtfilt(b_hf, a_hf, signal)

        x_lf = x_lf.transpose(1, 2, 0).reshape(N * C, T)
        x_bp = x_bp.transpose(1, 2, 0).reshape(N * C, T)
        x_hf = x_hf.transpose(1, 2, 0).reshape(N * C, T)

        x_lf = (x_lf - x_lf.mean()) / (x_lf.std() + 1e-8)
        x_bp = (x_bp - x_bp.mean()) / (x_bp.std() + 1e-8)
        x_hf = (x_hf - x_hf.mean()) / (x_hf.std() + 1e-8)
        return x_lf, x_bp, x_hf

    def _resolve_path(self, path_str):
        path = Path(path_str)
        if path.exists():
            return str(path)
        if self.base_dir and 'test_balanced_npz' in path.parts:
            idx = path.parts.index('test_balanced_npz')
            candidate = self.base_dir / Path(*path.parts[idx + 1:])
            if candidate.exists():
                return str(candidate)
        raise FileNotFoundError(f'Missing file: {path_str}')


In [ ]:
import torch

from torch.utils.data import DataLoader
from scipy import stats

sns.set_theme(style='whitegrid')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

split_json = project_dir / 'data_split.json'
base_dir = project_dir / 'test_balanced_npz'
output_root = project_dir / 'visualizations' / 'band_cut_grid_colab'
output_root.mkdir(parents=True, exist_ok=True)
print('Device:', device)

In [ ]:
def compute_bandwise_losses(model, loader, device):
    model.eval()
    losses, labels = [], []
    with torch.no_grad():
        for x_lf, x_bp, x_hf, batch_labels in loader:
            x_lf = x_lf.to(device)
            x_bp = x_bp.to(device)
            x_hf = x_hf.to(device)
            recons, _, _, _ = model(x_lf, x_bp, x_hf)
            mse_lf = torch.nn.functional.mse_loss(recons['lf'], x_lf, reduction='none').mean(dim=[1,2])
            mse_bp = torch.nn.functional.mse_loss(recons['bp'], x_bp, reduction='none').mean(dim=[1,2])
            mse_hf = torch.nn.functional.mse_loss(recons['hf'], x_hf, reduction='none').mean(dim=[1,2])
            losses.append(torch.stack([mse_lf, mse_bp, mse_hf], dim=1).cpu().numpy())
            labels.extend(batch_labels.numpy())
    return np.concatenate(losses), np.array(labels)

def extract_latents(model, loader, device):
    model.eval()
    latents, labels = [], []
    with torch.no_grad():
        for x_lf, x_bp, x_hf, batch_labels in loader:
            x_lf = x_lf.to(device)
            x_bp = x_bp.to(device)
            x_hf = x_hf.to(device)
            _, mus, _, _ = model(x_lf, x_bp, x_hf)
            z = torch.cat([mus['lf'].mean(dim=2), mus['bp'].mean(dim=2), mus['hf'].mean(dim=2)], dim=1)
            latents.append(z.cpu().numpy())
            labels.extend(batch_labels.numpy())
    return np.concatenate(latents), np.array(labels)

def fit_gaussian(data):
    mu = np.mean(data, axis=0)
    cov = np.cov(data, rowvar=False)
    return mu, cov

def within_confidence(x, mu, cov, confidence):
    try:
        diff = x - mu
        cov_inv = np.linalg.inv(cov + 1e-6 * np.eye(len(mu)))
        mahal = diff @ cov_inv @ diff
        threshold = stats.chi2.ppf(confidence, len(mu))
        return mahal <= threshold
    except np.linalg.LinAlgError:
        sigma = np.sqrt(np.diag(cov))
        z_score = stats.norm.ppf((1 + confidence) / 2)
        return np.all(np.abs(x - mu) <= z_score * sigma)

def train_for_pair(model, loader, epochs, lr, train_real_only):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        for x_lf, x_bp, x_hf, labels in loader:
            if train_real_only:
                mask = labels == 0
                if not mask.any():
                    continue
                x_lf, x_bp, x_hf = x_lf[mask], x_bp[mask], x_hf[mask]
            x_lf = x_lf.to(device)
            x_bp = x_bp.to(device)
            x_hf = x_hf.to(device)
            opt.zero_grad()
            recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
            loss, _ = band_split_vae_loss(recons, mus, logvars, {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}, x_hat_fused, None)
            loss.backward()
            opt.step()
    model.eval()
    return model

In [ ]:
# Configure grid
fc_low_values = [1.0, 2.0, 3.0, 4.0]
fc_high_values = [6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0]
batch_size = 64
num_workers = 0
max_pairs = None
epochs = 5
lr = 1e-4
train_real_only = True
config = Config(mode='full')

results = []
for fc_low in fc_low_values:
    for fc_high in fc_high_values:
        if fc_high <= fc_low:
            continue
        print(f'Pair fc_low={fc_low}, fc_high={fc_high}')
        train_ds = InlineStage2Dataset(split_json_path=split_json, split_name='stage2_train', T_fixed=config.T_fixed, fps=config.fps, use_acceleration=config.use_acceleration, use_angle=config.use_angle, use_angle_rate=config.use_angle_rate, fc_low=fc_low, fc_high=fc_high, filter_order=config.filter_order, random_crop=False, base_dir=base_dir)
        test_ds = InlineStage2Dataset(split_json_path=split_json, split_name='final_test', T_fixed=config.T_fixed, fps=config.fps, use_acceleration=config.use_acceleration, use_angle=config.use_angle, use_angle_rate=config.use_angle_rate, fc_low=fc_low, fc_high=fc_high, filter_order=config.filter_order, random_crop=False, base_dir=base_dir)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

        model = BandSplitVAE(C_in_per_band=config.C_in_per_band, C_h=config.C_h, C_z=config.C_z, dilations=config.dilations).to(device)
        model = train_for_pair(model, train_loader, epochs=epochs, lr=lr, train_real_only=train_real_only)

        train_losses, train_labels = compute_bandwise_losses(model, train_loader, device)
        test_losses, test_labels = compute_bandwise_losses(model, test_loader, device)
        train_latents, _ = extract_latents(model, train_loader, device)
        test_latents, _ = extract_latents(model, test_loader, device)

        real_mask = train_labels == 0
        loss_mu, loss_cov = fit_gaussian(train_losses[real_mask])
        pca = PCA(n_components=10)
        train_lat_pca = pca.fit_transform(train_latents[real_mask])
        test_lat_pca = pca.transform(test_latents)
        latent_mu, latent_cov = fit_gaussian(train_lat_pca)

        preds = []
        for loss_vec, latent_vec in zip(test_losses, test_lat_pca):
            pass1 = within_confidence(loss_vec, loss_mu, loss_cov, 0.95)
            pass2 = within_confidence(latent_vec, latent_mu, latent_cov, 0.70)
            preds.append(0 if pass1 and pass2 else 1)
        preds = np.array(preds)

        acc = accuracy_score(test_labels, preds)
        prec, rec, f1, _ = precision_recall_fscore_support(test_labels, preds, average='binary')
        results.append({'fc_low': fc_low, 'fc_high': fc_high, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1})

        out_dir = output_root / f'fc_{fc_low}_{fc_high}'
        out_dir.mkdir(parents=True, exist_ok=True)
        with open(out_dir / 'metrics.json', 'w') as f:
            import json
            json.dump(results[-1], f, indent=2)

        if max_pairs and len(results) >= max_pairs:
            break
    if max_pairs and len(results) >= max_pairs:
        break

print('Results:', results)